# Bybit Intra Arb01 - Latest 8h Uniform Orders

Read all columns from `uniform_orders` for `bybit-intra-arb01` through the local persist read server. The server is configured with `max_window_sec = 3600`, so the notebook pulls the latest 8 hours in 1 hour chunks and concatenates them into `bybit_orders_8h`.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts' / 'persist_read_client.py').exists():
    REPO_ROOT = Path('/home/ubuntu/crypto_mkt/mkt_signal')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.persist_read_client import PersistReadClient, iter_windows

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

BASE_URL = 'http://127.0.0.1:8822'
SOURCE_ID = 'bybit-intra-arb01'
TABLE = 'uniform_orders'
WINDOW_HOURS = 8
CHUNK_SECONDS = 3600

client = PersistReadClient(BASE_URL, timeout_sec=120, tz='UTC')

## Health And Schema

In [ ]:
print(client.health())
schema = client.schema(TABLE, source_id=SOURCE_ID)
print(f'source_id={SOURCE_ID} table={schema.table} columns={len(schema.columns)} formats={schema.formats}')
list(schema.columns)

## Read Latest 8 Hours

In [ ]:
end = datetime.now(timezone.utc)
start = end - timedelta(hours=WINDOW_HOURS)

print('UTC window:', start.isoformat(), '->', end.isoformat())

frames = []
chunk_counts = []
for window in iter_windows(start, end, window_sec=CHUNK_SECONDS, tz=timezone.utc):
    chunk_start = datetime.fromtimestamp(window.start_us / 1_000_000, timezone.utc)
    chunk_end = datetime.fromtimestamp(window.end_us / 1_000_000, timezone.utc)
    df = client.read_pandas(
        TABLE,
        start_us=window.start_us,
        end_us=window.end_us,
        source_id=SOURCE_ID,
        columns=None,  # None means all columns from uniform_orders.
        timeout_sec=120,
    )
    frames.append(df)
    chunk_counts.append({
        'start_utc': chunk_start,
        'end_utc': chunk_end,
        'rows': len(df),
    })

bybit_orders_8h = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if 'ts_us' in bybit_orders_8h.columns:
    bybit_orders_8h['ts'] = pd.to_datetime(bybit_orders_8h['ts_us'], unit='us', utc=True)
    bybit_orders_8h = bybit_orders_8h.sort_values('ts_us').reset_index(drop=True)
if not bybit_orders_8h.empty:
    bybit_orders_8h.insert(0, 'source_id', SOURCE_ID)

print(f'rows={len(bybit_orders_8h):,} columns={len(bybit_orders_8h.columns)}')
pd.DataFrame(chunk_counts)

## Preview

In [ ]:
bybit_orders_8h.head(20)

In [ ]:
bybit_orders_8h.tail(20)

## Quick Checks

In [ ]:
if bybit_orders_8h.empty:
    pd.DataFrame(columns=['status', 'orders'])
else:
    bybit_orders_8h.groupby('status', dropna=False).size().reset_index(name='orders').sort_values('orders', ascending=False)

In [ ]:
if bybit_orders_8h.empty:
    pd.DataFrame(columns=['symbol', 'orders'])
else:
    bybit_orders_8h.groupby('symbol', dropna=False).size().reset_index(name='orders').sort_values('orders', ascending=False).head(30)

## Full DataFrame

`bybit_orders_8h` contains the full latest 8 hour order dataset. Jupyter may truncate display, but the dataframe in memory has all rows and columns.

In [ ]:
bybit_orders_8h

## Optional Export

In [ ]:
# Set to True when you want a local parquet snapshot under order_exports/.
EXPORT_PARQUET = False

if EXPORT_PARQUET:
    export_dir = REPO_ROOT / 'order_exports' / SOURCE_ID / 'latest_8h'
    export_dir.mkdir(parents=True, exist_ok=True)
    export_path = export_dir / f'{TABLE}_{start:%Y%m%dT%H%M%SZ}_{end:%Y%m%dT%H%M%SZ}.parquet'
    bybit_orders_8h.to_parquet(export_path, index=False)
    print(export_path)
else:
    print('Set EXPORT_PARQUET = True to write a parquet snapshot.')